<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets kagglehub pandas

In [2]:
# 1. Load the "MongoDB/whatscooking.restaurants" from HF

from datasets import load_dataset

hf_dataset = load_dataset("MongoDB/whatscooking.restaurants")

README.md:   0%|          | 0.00/5.78k [00:00<?, ?B/s]

whatscooking.restaurants.json: reconstructing file:   0%|          |  0.00B /  147MB            

whatscooking.restaurants.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25361 [00:00<?, ? examples/s]

In [4]:
for split_name, split_rows in hf_dataset.items():
    hf_df = split_rows.to_pandas()
    hf_df.to_csv(f"/content/mongodb_restaurants_{split_name}.csv", index = False)

In [5]:
# 2. Load the "Spam Text Message Classification" from Kaggle

import kagglehub

kaggle_df = kagglehub.dataset_load(kagglehub.KaggleDatasetAdapter.PANDAS,   "team-ai/spam-text-message-classification", "SPAM text message 20170820 - Data.csv")

100%|██████████| 474k/474k [00:00<00:00, 500kB/s]


In [6]:
kaggle_df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [7]:
kaggle_df.to_csv("/content/email_spam_classification.csv", index = False)

In [10]:
# What is a `*,jsonl` file
# {... } row of dataset
# {... } row of dataset
# {... } row of dataset

kaggle_df.to_json("/content/email_spam_classification.jsonl", index = False, orient="records", lines = True)

In [12]:
# Fine-Tuning model using jsonl scheme: {{ "messages ": [{ "role": "system", "content": "instructions"}, {"role":"user", "content": "..."}]}}

SYSTEM_MESSAGE = "Classify the message strictly as 'spam' or 'ham'. Return only one word."

fine_tuning_df = kaggle_df.apply(
    lambda x: [
        { "role": "developer", "content": SYSTEM_MESSAGE},
        { "role": "user", "content": f"Classify the message: \"{x["Message"]}\""},
        { "role": "assistant", "content": x["Category"]} # очакван резултат
    ],
    axis = 1
)

fine_tuning_df = fine_tuning_df.to_frame(name="messages")

In [13]:
fine_tuning_df

,messages
0,"[{'role': 'developer', 'content': 'Classify th..."
1,"[{'role': 'developer', 'content': 'Classify th..."
2,"[{'role': 'developer', 'content': 'Classify th..."
3,"[{'role': 'developer', 'content': 'Classify th..."
4,"[{'role': 'developer', 'content': 'Classify th..."
...,...
5567,"[{'role': 'developer', 'content': 'Classify th..."
5568,"[{'role': 'developer', 'content': 'Classify th..."
5569,"[{'role': 'developer', 'content': 'Classify th..."
5570,"[{'role': 'developer', 'content': 'Classify th..."


In [14]:
fine_tuning_df.to_json("/content/fine_tuning.jsonl", index = False, orient="records", lines = True)